 # Workflow for a transformation pathway of a single node energy system with perfect foresight

 In this application of the ETHOS.FINE framework, a transformation pathway of a energy system is modeled and optimized.

 All classes which are available to the user are utilized and examples of the selection of different parameters within these classes are given.

 The workflow is structures as follows:
 1. Required packages are imported and the input data path is set
 2. An energy system model instance is created
 3. Commodity sources are added to the energy system model
 4. Commodity conversion components are added to the energy system model
 5. Commodity storages are added to the energy system model
 6. Commodity sinks are added to the energy system model
 7. Material sinks are added to the energy system model
 8. Material sources are addeed to the energy system model
 9. Material conversions are added to the energy system model for recycling processes
 10. The energy system model is optimized
 11. Selected optimization results are presented


 # 1. Import required packages and set input data path

 The ETHOS.FINE framework is imported which provides the required classes and functions for modeling the energy system.

In [1]:
import fine as fn
from getData import getData
from pathlib import Path
import pandas as pd


cwd = Path.cwd()
data = getData()

 # 2. Create an energy system model instance

 The structure of the energy system model is given by the considered locations, commodities, the number of time steps as well as the hours per time step.

 The commodities are specified by a unit (i.e. 'GW_electric', 'GW_H2lowerHeatingValue', 'Mio. t CO2/h') which can be given as an energy or mass unit per hour. Furthermore, the cost unit and length unit are specified.

In [2]:
locations = {"GermanyRegion"}
commodityUnitsDict = {"electricity": r"GW$_{el}$", "hydrogen": r"GW$_{H_{2},LHV}$"}
commodities = {"electricity", "hydrogen"}
materials = {"steel", "copper", "copper_unref", "ore"}
materialUnitsDict = {
    "steel": "tons/h",
    "copper": "tons/h",
    "copper_unref": "tons/h",
    "ore": "tons/h",
}

numberOfTimeSteps = 8760
hoursPerTimeStep = 1

In [3]:
initial_material_cost = {
    "steel": pd.Series(
        {
            "GermanyRegion": 0.1,
        }
    ),
    "copper": pd.Series(
        {
            "GermanyRegion": 0.1,
        }
    ),
}

 # 2.1 define Transformation Pathway parameters

 Transformation Pathway Analyses can be run by setting a number of investment periods
 larger than 1, which is the default value and results in a single year optimization.

In [4]:
numberOfInvestmentPeriods = 3
startYear = 2020
interval = 5

In [5]:
# pathwayBalanceLimit = pd.DataFrame(
#    columns=["GermanyRegion", "Total", "lowerBound"],
#    index=["Copper_Resources", "Steel_Resources"],
# )
# pathwayBalanceLimit.loc["Copper_Resources"] = [None, 650, False]
# pathwayBalanceLimit.loc["Steel_Resources"] = [None, 650, False]

In [6]:
esM = fn.EnergySystemModel(
    locations=locations,
    commodities=commodities,
    materials=materials,
    numberOfInvestmentPeriods=numberOfInvestmentPeriods,
    startYear=startYear,
    investmentPeriodInterval=interval,
    numberOfTimeSteps=8760,
    commodityUnitsDict=commodityUnitsDict,
    materialUnitsDict=materialUnitsDict,
    hoursPerTimeStep=1,
    costUnit="1e9 Euro",
    lengthUnit="km",
    verboseLogLevel=0,
    # pathwayBalanceLimit=pathwayBalanceLimit,
    initialMaterialCost=initial_material_cost,

)

 # 3. Add commodity sources to the energy system model

 ## 3.1. Electricity sources

 ### Wind onshore

 change weather conditions for the different investment periods

In [7]:
operationRateMax = {}
operationRateMax[2020] = 1.2 * data["Wind (onshore), operationRateMax"]
operationRateMax[2025] = 0.7 * data["Wind (onshore), operationRateMax"]
operationRateMax[2030] = 1 * data["Wind (onshore), operationRateMax"]

 define existing stock for wind onshore turbines

In [8]:
stockWindonshoreCommissioning = {
    2015: 0,
}

stockWindoffshoreCommissioning = {
    2015: 0,
}

 define invest and opex per capacity for wind onshore turbines

 add wind onshore source to esM

In [9]:
esM.add(
    fn.Source(
        esM=esM,
        name="windonshore",
        commodity="electricity",
        hasCapacityVariable=True,
        operationRateMax=data["Wind (onshore), operationRateMax"],
        capacityMax=data["Wind (onshore), capacityMax"],
        investPerCapacity=3.1,
        opexPerCapacity=3.1 * 0.02,
        interestRate=0.08,
        economicLifetime=5,
        stockCommissioning=stockWindonshoreCommissioning,
        materialIntensity={
            2015: {  # IP für 2015
                "copper": pd.Series({"GermanyRegion": 5.1}),
            },
            2020: {  # IP für 2020
                "copper": pd.Series({"GermanyRegion": 3.1}),
            },
            2025: {  # IP für 2025
                "copper": pd.Series({"GermanyRegion": 3.0}),
            },
            2030: {  # IP für 2030
                "copper": pd.Series({"GermanyRegion": 2.9}),
            },
        },
        materialCollection={
            2020: {  # IP für 2020
                "copper": pd.Series({"GermanyRegion": 0.9}),
            },
            2025: {  # IP für 2025
                "copper": pd.Series({"GermanyRegion": 0.9}),
            },
            2030: {  # IP für 2030
                "copper": pd.Series({"GermanyRegion": 0.9}),
            },
        },
    )
)

In [10]:
esM.add(
    fn.Source(
        esM=esM,
        name="windoffshore",
        commodity="electricity",
        hasCapacityVariable=True,
        operationRateMax=data["Wind (onshore), operationRateMax"],
        capacityMax=data["Wind (onshore), capacityMax"],
        investPerCapacity=3.1,
        opexPerCapacity=3.1 * 0.02,
        interestRate=0.08,
        economicLifetime=5,
        stockCommissioning=stockWindonshoreCommissioning,
        materialIntensity={
            2015: {  # IP für 2015
                "steel": pd.Series({"GermanyRegion": 4.1}),
            },
            2020: {  # IP für 2020
                "steel": pd.Series({"GermanyRegion": 3.1}),
            },
            2025: {  # IP für 2025
                "steel": pd.Series({"GermanyRegion": 3.0}),
            },
            2030: {  # IP für 2030
                "steel": pd.Series({"GermanyRegion": 2.9}),
            },
        },
        materialCollection={
            2020: {  # IP für 2020
                "steel": pd.Series({"GermanyRegion": 0.9}),
            },
            2025: {  # IP für 2025
                "steel": pd.Series({"GermanyRegion": 0.9}),
            },
            2030: {  # IP für 2030
                "steel": pd.Series({"GermanyRegion": 0.9}),
            },
        },
    )
)

 Full load hours:

In [11]:
data["Wind (onshore), operationRateMax"].sum()

2300.4069071646272

 # 4. Add conversion components to the energy system model

 ### Electrolyzers

 add component with constant invest and opex per capacity

In [12]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="Electroylzers",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={"electricity": -1, "hydrogen": 0.7},
        hasCapacityVariable=True,
        investPerCapacity=0.5,
        opexPerCapacity=0.5 * 0.025,
        interestRate=0.08,
        economicLifetime=5,
    )
)

 # 5. Add commodity storages to the energy system model

 ## 5.1. Electricity storage

 ### Lithium ion batteries

 The self discharge of a lithium ion battery is here described as 3% per month. The self discharge per hours is obtained using the equation (1-$\text{selfDischarge}_\text{hour})^{30*24\text{h}} = 1-\text{selfDischarge}_\text{month}$.

In [13]:
esM.add(
    fn.Storage(
        esM=esM,
        name="Li-ion batteries",
        commodity="electricity",
        hasCapacityVariable=True,
        chargeEfficiency=0.95,
        cyclicLifetime=10000,
        dischargeEfficiency=0.95,
        selfDischarge=1 - (1 - 0.03) ** (1 / (30 * 24)),
        chargeRate=1,
        dischargeRate=1,
        doPreciseTsaModeling=False,
        investPerCapacity=0.151,
        opexPerCapacity=0.002,
        interestRate=0.08,
        economicLifetime=15,
    )
)

 ## 5.2. Hydrogen storage

 ### Hydrogen filled salt caverns
 The maximum capacity is here obtained by: dividing the given capacity (which is given for methane) by the lower heating value of methane and then multiplying it with the lower heating value of hydrogen.

In [14]:
esM.add(
    fn.Storage(
        esM=esM,
        name="Salt caverns (hydrogen)",
        commodity="hydrogen",
        hasCapacityVariable=True,
        capacityVariableDomain="continuous",
        capacityPerPlantUnit=133,
        chargeRate=1 / 470.37,
        dischargeRate=1 / 470.37,
        sharedPotentialID="Existing salt caverns",
        stateOfChargeMin=0.33,
        stateOfChargeMax=1,
        capacityMax=data["Salt caverns (hydrogen), capacityMax"],
        investPerCapacity={2020: 0.00011, 2025: 0.00009, 2030: 0.00009},
        opexPerCapacity=0.00057,
        interestRate=0.08,
        economicLifetime=30,
    )
)

 # 6. Add commodity sinks to the energy system model

 ## 6.1. Electricity sinks

 ### Electricity demand

 vary the demand with the years - increasing demand by 30% per year

In [15]:
electricityDemand = {}
electricityDemand[2020] = (1 + 0 * 0.3) * data["Electricity demand, operationRateFix"]
electricityDemand[2025] = (1 + 1 * 0.3) * data["Electricity demand, operationRateFix"]
electricityDemand[2030] = (1 + 2 * 0.3) * data["Electricity demand, operationRateFix"]

esM.add(
    fn.Sink(
        esM=esM,
        name="Electricity demand",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=electricityDemand,
    )
)

 ## 6.2. Hydrogen sinks

 ### Fuel cell electric vehicle (FCEV) demand

In [16]:
FCEV_penetration = 0.5

# vary the demand with the years - increasing demand by 25% per year
hydrogendDemand = {}
hydrogendDemand[2020] = (
    (1 + 0 * 0.25) * data["Hydrogen demand, operationRateFix"] * FCEV_penetration
)
hydrogendDemand[2025] = (
    (1 + 0 * 0.25) * data["Hydrogen demand, operationRateFix"] * FCEV_penetration
)
hydrogendDemand[2030] = (
    (1 + 0 * 0.25) * data["Hydrogen demand, operationRateFix"] * FCEV_penetration
)


esM.add(
    fn.Sink(
        esM=esM,
        name="Hydrogen demand",
        commodity="hydrogen",
        hasCapacityVariable=False,
        operationRateFix=hydrogendDemand,
    )
)

# 7. Add material sinks to the energy system model

In [17]:
sink = esM.add(
    fn.Sink(
        esM=esM,
        name="Steel demand",
        hasCapacityVariable=False,
        commodity="steel",
        material=True,
    )
)

sink = esM.add(
    fn.Sink(
        esM=esM,
        name="Copper demand",
        hasCapacityVariable=False,
        commodity="copper",
        material=True,
    )
)

In [18]:
esM.generationMaterialSinks()

Existing material sinks: {'steel', 'copper'}
Missing materials sinks: {'ore', 'copper_unref'}
New sink added: Ore demand
New sink added: Copper_unref demand


# 8. Add material sources to the energy system model


## 8.1 Add primary material sources with pathway balance limit


In [19]:
esM.add(
    fn.Source(
        esM=esM,
        name="Ore supply",
        hasCapacityVariable=True,
        commodity="ore",
    )
)


esM.add(
    fn.Conversion(
        esM=esM,
        name="Ore conversion",
        physicalUnit=r"tons/h",
        commodityConversionFactors={"ore": -1, "steel": 0.8, "copper_unref": 0.2},
        hasCapacityVariable=True,
        economicLifetime=35,
    )
)

esM.add(
    fn.Conversion(
        esM=esM,
        name="Copper conversion",
        physicalUnit=r"tons/h",
        commodityConversionFactors={"copper_unref": -1, "copper": 1},
        hasCapacityVariable=True,
        economicLifetime=35,
    )
)


esM.add(
    fn.Sink(
        esM=esM,
        name="Copper overproduction",
        hasCapacityVariable=False,
        commodity="copper",
    )
)

In [20]:
esM.add(
    fn.Source(
        esM=esM,
        name="windoffshore_steel_scrap",
        commodity="windoffshore_steel_scrap",
        hasCapacityVariable=False,
        material=True,
    )
)

esM.add(
    fn.Sink(
        esM=esM,
        name="windoffshore_steel_scrap rec",
        hasCapacityVariable=False,
        commodity="windoffshore_steel_scrap",
    )
)

esM.add(
    fn.Source(
        esM=esM,
        name="windonshore_copper_scrap",
        commodity="windonshore_copper_scrap",
        hasCapacityVariable=False,
        material=True,
    )
)

esM.add(
    fn.Sink(
        esM=esM,
        name="windonshore_copper_scrap rec",
        hasCapacityVariable=False,
        commodity="windonshore_copper_scrap",
    )
)

# 9. Add recycling plants

In [21]:
esM.add(
    fn.Conversion(
        esM=esM,
        name="Recycler Onshore",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={
            "windonshore_copper_scrap": -1,
            "copper": 0.4,
        },
        hasCapacityVariable=True,
        investPerCapacity=0.7,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=33,
    )
)

esM.add(
    fn.Conversion(
        esM=esM,
        name="Recycler Offshore",
        physicalUnit=r"GW$_{el}$",
        commodityConversionFactors={
            "windoffshore_steel_scrap": -1,
            "steel": 0.5,
        },
        hasCapacityVariable=True,
        investPerCapacity=0.7,
        opexPerCapacity=0.021,
        interestRate=0.08,
        economicLifetime=33,
    )
)

 # 10. Optimize energy system model

 All components are now added to the model and the model can be optimized. If the computational complexity of the optimization should be reduced, the time series data of the specified components can be clustered before the optimization and the parameter timeSeriesAggregation is set to True in the optimize call.

In [22]:
esM.aggregateTemporally(numberOfTypicalPeriods=30)


Clustering time series data with 30 typical periods and 24 time steps per period 
further clustered to 12 segments per period...
		(16.4456 sec)



In [23]:
esM.optimize(timeSeriesAggregation=True, solver="gurobi")

Time series aggregation specifications:
Number of typical periods:30, number of time steps per period:24, number of segments per period:12

Declaring sets, variables and constraints for SourceSinkModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.8642 sec)

Declaring sets, variables and constraints for ConversionModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.4607 sec)

Declaring sets, variables and constraints for StorageModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(2.0037 sec)

		(0.0003 sec)

Declaring shared potential constraint...
		(0.0016 sec)

Declaring linked component quantity constraint...
		(0.0000 sec)

Declaring commodity balances...
		(0.5554 sec)

Declaring material demand constraints...
RHS_demand 3.1*commis_srcSnk[GermanyRegion,windoffshore,0]
RHS_demand 3.0*commis_srcSnk[GermanyRegion,windoffshore,1]
LHS_demand 75*op_srcSnk[GermanyRegion,Steel demand,0,0,0] + 75*op_s

/fast/home/l-soeltzer/code/fine/fine/storage.py:2036: UserWarning: Charge and discharge at the same time for component Li-ion batteries
  warnings.warn(
/fast/home/l-soeltzer/code/fine/fine/storage.py:2036: UserWarning: Charge and discharge at the same time for component Salt caverns (hydrogen)
  warnings.warn(
/fast/home/l-soeltzer/code/fine/fine/storage.py:2036: UserWarning: Charge and discharge at the same time for component Li-ion batteries
  warnings.warn(
/fast/home/l-soeltzer/code/fine/fine/storage.py:2036: UserWarning: Charge and discharge at the same time for component Salt caverns (hydrogen)
  warnings.warn(
/fast/home/l-soeltzer/code/fine/fine/storage.py:2036: UserWarning: Charge and discharge at the same time for component Li-ion batteries
  warnings.warn(
/fast/home/l-soeltzer/code/fine/fine/storage.py:2036: UserWarning: Charge and discharge at the same time for component Salt caverns (hydrogen)
  warnings.warn(


for StorageModel ...  (2.9013sec)
		(6.9446 sec)



 # 11. Selected results output

In [24]:
total_cost = 0.0

for loc, mat in esM.pyM.initialMaterialSet:
    quantity = esM.pyM.initialMaterialSupply[loc, mat].value
    unit_cost = esM.initialMaterialCost[mat][loc]
    material_cost = quantity * unit_cost

    print(f"{mat}:")
    print(f"  Initial supply [tons]: {quantity:.6f}")
    print(f"  Unit cost: {unit_cost:.8f} [USD/tons]")
    print(f"  Cost contribution : {material_cost:.6f} [USD]")

    total_cost += material_cost

print(f"\nTotal cost initial supply: {total_cost:.6f} [USD]")

steel:
  Initial supply [tons]: 84.581616
  Unit cost: 0.10000000 [USD/tons]
  Cost contribution : 8.458162 [USD]
copper:
  Initial supply [tons]: 0.000000
  Unit cost: 0.10000000 [USD/tons]
  Cost contribution : 0.000000 [USD]

Total cost initial supply: 8.458162 [USD]


In [25]:
import pyomo.environ as pyomo

initial_cost = sum(
    esM.pyM.initialMaterialSupply[loc, mat] * esM.initialMaterialCost[mat][loc]
    for loc, mat in esM.pyM.initialMaterialSet
)

print("Initial material cost:", pyomo.value(initial_cost))
print("Total Objective:", pyomo.value(esM.pyM.Obj))

Initial material cost: 8.45816162277569
Total Objective: 424.2817335375773


 ### Sources and Sink

 Show optimization summary

In [26]:
for year in [2020, 2025, 2030]:
    print(f"\n Results of SourceSinkModel for year {year}")
    print(esM.getOptimizationSummary("SourceSinkModel", outputLevel=2, ip=year))


 Results of SourceSinkModel for year 2020
                                                             GermanyRegion
Component             Property        Unit                                
Copper overproduction operation       [tons/h*h/a]                5.384147
                                      [tons/h*h]                  5.384147
Electricity demand    operation       [GW$_{el}$*h/a]         30957.888055
                                      [GW$_{el}$*h]           30957.888055
Hydrogen demand       operation       [GW$_{H_{2},LHV}$*h/a]   4765.074877
                                      [GW$_{H_{2},LHV}$*h]     4765.074877
Ore supply            capacity        [tons/h]                    0.598239
                      commissioning   [tons/h]                    0.598239
                      operation       [tons/h*h/a]               26.920734
                                      [tons/h*h]                 26.920734
Steel demand          operation       [tons/h*h/a]       

 ### Conversion

 Show optimization summary

In [27]:
for year in [2020, 2025, 2030]:
    print(f"\n Results of ConversionMpdel for year {year}")
    print(esM.getOptimizationSummary("ConversionModel", outputLevel=2, ip=year))


 Results of ConversionMpdel for year 2020
                                                  GermanyRegion
Component         Property        Unit                         
Copper conversion capacity        [tons/h]             0.119648
                  commissioning   [tons/h]             0.119648
                  operation       [tons/h*h/a]         5.384147
                                  [tons/h*h]           5.384147
Electroylzers     NPVcontribution [1e9 Euro]           1.678316
                  TAC             [1e9 Euro/a]         0.389208
                  capacity        [GW$_{el}$]          2.825916
                  capexCap        [1e9 Euro/a]         0.353884
                  commissioning   [GW$_{el}$]          2.825916
                  invest          [1e9 Euro]           1.412958
                  operation       [GW$_{el}$*h/a]   6807.249824
                                  [GW$_{el}$*h]     6807.249824
                  opexCap         [1e9 Euro/a]         0.0353

 ### Storage

 Show optimization summary

In [28]:
for year in [2020, 2025, 2030]:
    print(f"\n Results of StorageModel for year {year}")
    print(esM.getOptimizationSummary("StorageModel", outputLevel=2, ip=year))


 Results of StorageModel for year 2020
                                                                  GermanyRegion
Component               Property           Unit                                
Li-ion batteries        NPVcontribution    [1e9 Euro]                 56.823352
                        TAC                [1e9 Euro/a]                13.17757
                        capacity           [GW$_{el}$*h]             670.912595
                        capexCap           [1e9 Euro/a]               11.835744
                        commissioning      [GW$_{el}$*h]             670.912595
                        invest             [1e9 Euro]                101.307802
                        operationCharge    [GW$_{el}$*h/a]          31795.29795
                                           [GW$_{el}$*h]            31795.29795
                        operationDischarge [GW$_{el}$*h/a]          28575.78213
                                           [GW$_{el}$*h]            28575.78213
